# GDELT Data
EDA approaches to gathering energy based sources from GDELT

### Imports and Notebook Settings

In [1]:
import os 
import re
from pathlib import Path

In [2]:
import pandas as pd
import numpy as np 

import matplotlib.pyplot as plt
# import seaborn as sns 
import plotly.express as px
import pprint 

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
ROOT = Path.cwd().parent

In [5]:
style_path = ( ROOT 
              / 'notebooks' 
              / 'styler.mplstyle'
              )
plt.style.use(style_path)

In [6]:
import requests

### Load Data

In [7]:
DATA_PATH = ROOT / 'data' / 'samples' / 'gkg_sample_2025-01-01_to_2025-12-31_3days_per_month_4files_per_day.parquet'

In [8]:
df = pd.read_parquet(DATA_PATH)

In [9]:
len(df)

171090

In [10]:
df.head(3)

,GKGRECORDID,DATE,SourceCollectionIdentifier,SourceCommonName,DocumentIdentifier,Counts,V2Counts,Themes,V2Themes,Locations,...,RelatedImages,SocialImageEmbeds,SocialVideoEmbeds,Quotations,AllNames,Amounts,TranslationInfo,Extras,GDELT_TIMESTAMP,GDELT_URL
0,20250101000000-0,20250101000000,1,crooksandliars.com,https://crooksandliars.com/2024/12/retired-gen...,None,None,CRISISLEX_C07_SAFETY;WB_2470_PEACE_OPERATIONS_...,"TAX_MILITARY_TITLE_LIEUTENANT,90;TAX_MILITARY_...","4#Shanghai, Shanghai, China#CH#CH23#31.2222#12...",...,None,None,https://youtube.com/embed/3q5NHISy1G0;,None,"Elon Musk,10;Lieutenant General Russel,108;Chi...",None,None,<PAGE_LINKS>https://www.independent.co.uk/news...,20250101000000,http://data.gdeltproject.org/gdeltv2/202501010...
1,20250101000000-1,20250101000000,1,weartv.com,https://weartv.com/news/nation-world/court-doc...,"ARREST#100##3#Fayette County, Alabama, United ...","ARREST#100##3#Fayette County, Alabama, United ...",AFFECT;TAX_FNCACT;TAX_FNCACT_CHILD;TAX_FNCACT_...,"TAX_FNCACT_CHILD,295;TAX_FNCACT_CHILD,1272;TAX...","2#Alabama, United States#US#USAL#32.799#-86.80...",...,None,None,https://youtube.com/c/NewsChannel8Tulsa/videos;,None,"Wendy Pamela Bailey,212;Kahleb Rowan Collins,3...","2,occasions,434;3,weeks,1585;100,dollars ,1688;",None,<PAGE_AUTHORS>CYNTHIA GOULD;WBMA staff</PAGE_A...,20250101000000,http://data.gdeltproject.org/gdeltv2/202501010...
2,20250101000000-2,20250101000000,1,yahoo.com,https://www.yahoo.com/news/fbi-says-seized-big...,None,None,TAX_FNCACT;TAX_FNCACT_MAN;ARREST;SOC_GENERALCR...,"TAX_WEAPONS_EXPLOSIVES,150;TAX_FNCACT_MAN,14;T...",None,...,None,None,https://youtube.com/yahoo;,None,"Ken Dilanian,189",None,None,<PAGE_PRECISEPUBTIMESTAMP>20241231230300</PAGE...,20250101000000,http://data.gdeltproject.org/gdeltv2/202501010...


In [11]:
df['DATETIME'] = pd.to_datetime(df['DATE'])

### General Data Structure

In [12]:
def parse_multi(value: str) -> set[str]:
    if pd.isna(value):
        return set()

    return {
        theme.strip().upper()
        for theme in value.split(";")
        if theme.strip()
    }



In [13]:
rows = len(df)
cols = len(df.iloc[0])

print(f"Dataframe has dimension of {rows} Rows and {cols} Cols")

Dataframe has dimension of 171090 Rows and 30 Cols


In [14]:
pprint.pprint(list(df.columns))

['GKGRECORDID',
 'DATE',
 'SourceCollectionIdentifier',
 'SourceCommonName',
 'DocumentIdentifier',
 'Counts',
 'V2Counts',
 'Themes',
 'V2Themes',
 'Locations',
 'V2Locations',
 'Persons',
 'V2Persons',
 'Organizations',
 'V2Organizations',
 'Tone',
 'Dates',
 'GCAM',
 'SharingImage',
 'RelatedImages',
 'SocialImageEmbeds',
 'SocialVideoEmbeds',
 'Quotations',
 'AllNames',
 'Amounts',
 'TranslationInfo',
 'Extras',
 'GDELT_TIMESTAMP',
 'GDELT_URL',
 'DATETIME']


In [15]:
print("Perecentage of Missing Values")
df.isna().sum() / len(df)

Perecentage of Missing Values


GKGRECORDID                   0.000000
DATE                          0.000000
SourceCollectionIdentifier    0.000000
SourceCommonName              0.000000
DocumentIdentifier            0.000000
Counts                        0.876357
V2Counts                      0.876357
Themes                        0.109673
V2Themes                      0.109679
Locations                     0.261009
V2Locations                   0.262090
Persons                       0.203706
V2Persons                     0.211059
Organizations                 0.260290
V2Organizations               0.289386
Tone                          0.000000
Dates                         0.635759
GCAM                          0.000000
SharingImage                  0.126267
RelatedImages                 0.875738
SocialImageEmbeds             0.882986
SocialVideoEmbeds             0.444141
Quotations                    0.774154
AllNames                      0.044731
Amounts                       0.259489
TranslationInfo          

In [16]:
print("First Row")

df.iloc[0]

First Row


GKGRECORDID                                                    20250101000000-0
DATE                                                             20250101000000
SourceCollectionIdentifier                                                    1
SourceCommonName                                             crooksandliars.com
DocumentIdentifier            https://crooksandliars.com/2024/12/retired-gen...
Counts                                                                     None
V2Counts                                                                   None
Themes                        CRISISLEX_C07_SAFETY;WB_2470_PEACE_OPERATIONS_...
V2Themes                      TAX_MILITARY_TITLE_LIEUTENANT,90;TAX_MILITARY_...
Locations                     4#Shanghai, Shanghai, China#CH#CH23#31.2222#12...
V2Locations                   4#Shanghai, Shanghai, China#CH#CH23#13243#31.2...
Persons                                                               elon musk
V2Persons                               

#### Locations

In [17]:
locations = (
    df["Locations"]
    .dropna()
    .str.split(";")
    .explode()
    .reset_index()
)

locations.head()

,index,Locations
0,0,"4#Shanghai, Shanghai, China#CH#CH23#31.2222#12..."
1,0,"2#New York, United States#US#USNY#42.1497#-74...."
2,0,1#China#CH#CH#35#105#CH
3,1,"2#Alabama, United States#US#USAL#32.799#-86.80..."
4,1,"3#Fayette County, Alabama, United States#US#US..."


In [18]:
location_parts = locations["Locations"].str.split("#", expand=True)

location_parts.columns = [
    "location_type",
    "location_name",
    "country_code",
    "adm1_code",
    "latitude",
    "longitude",
    "feature_id",
    "extra",
]

location_parts = pd.concat(
    [locations[["index"]], location_parts],
    axis=1
)

In [19]:
location_parts

,index,location_type,location_name,country_code,adm1_code,latitude,longitude,feature_id,extra
0,0,4,"Shanghai, Shanghai, China",CH,CH23,31.2222,121.458,-1924465,None
1,0,2,"New York, United States",US,USNY,42.1497,-74.9384,NY,None
2,0,1,China,CH,CH,35,105,CH,None
3,1,2,"Alabama, United States",US,USAL,32.799,-86.8073,AL,None
4,1,3,"Fayette County, Alabama, United States",US,USAL,33.7334,-87.7167,161554,None
...,...,...,...,...,...,...,...,...,...
493254,171086,3,"White House, District Of Columbia, United States",US,USDC,38.8951,-77.0364,531871,None
493255,171086,1,United States,US,US,39.828175,-98.5795,US,None
493256,171086,3,"National Trust For Historic Preservation, Dist...",US,USDC,38.9093,-77.0419,530124,None
493257,171086,2,"Florida, United States",US,USFL,27.8333,-81.717,FL,None


In [20]:
(location_parts['country_code'].value_counts() / len(location_parts)).head(5)

country_code
US    0.512623
UK    0.062774
AS    0.054428
CA    0.050943
IN    0.022400
Name: count, dtype: float64

In [21]:
location_parts["location_name"].value_counts().head(20)

location_name
United States                                       35163
Washington, Washington, United States               13409
United Kingdom                                      12925
New York, United States                             12334
Canada                                              10069
Australia                                            9761
California, United States                            8105
Texas, United States                                 7551
China                                                7375
White House, District Of Columbia, United States     7327
London, London, City Of, United Kingdom              5468
Russia                                               5440
Florida, United States                               5173
France                                               4588
Israel                                               4236
Ukraine                                              3882
Japan                                                3807


In [22]:
city_names = location_parts.loc[(location_parts['country_code'] == 'US') & (location_parts['location_name'] != 'United States'), 'location_name']

In [23]:
city_names.value_counts().head(20)

location_name
Washington, Washington, United States               13409
New York, United States                             12334
California, United States                            8105
Texas, United States                                 7551
White House, District Of Columbia, United States     7327
Florida, United States                               5173
Chicago, Illinois, United States                     3204
Michigan, United States                              2835
Minnesota, United States                             2780
Hollywood, California, United States                 2777
Georgia, United States                               2574
Virginia, United States                              2454
Ohio, United States                                  2342
Illinois, United States                              2329
Tennessee, United States                             2327
Los Angeles, California, United States               2303
Pennsylvania, United States                          2175


#### URLS

In [24]:
df['SourceCommonName'].value_counts().head(10)

SourceCommonName
iheart.com         13364
yahoo.com           6351
tvguide.co.uk       2463
biztoc.com          1247
justjared.com        755
indiatimes.com       754
dailymail.co.uk      677
miragenews.com       668
amazonaws.com        472
thestar.com.my       461
Name: count, dtype: int64

#### Themes

In [25]:
Themes = (df['Themes']
          .str
          .strip()
          .str
          .split(';')
          .explode()
          .reset_index()
) 

In [26]:
for i in Themes:
    print(i)

index
Themes


In [27]:
Themes.value_counts().head()

index   Themes                   
81111   SOC_POINTSOFINTEREST_JAIL    3
138645  TAX_FNCACT_ACTIVISTS         3
131614  SOC_POINTSOFINTEREST_JAIL    3
96997   SOC_POINTSOFINTEREST_JAIL    3
116264  SOC_POINTSOFINTEREST_JAIL    2
Name: count, dtype: int64

#### Names and Organizations

In [28]:
df.columns

Index(['GKGRECORDID', 'DATE', 'SourceCollectionIdentifier', 'SourceCommonName',
       'DocumentIdentifier', 'Counts', 'V2Counts', 'Themes', 'V2Themes',
       'Locations', 'V2Locations', 'Persons', 'V2Persons', 'Organizations',
       'V2Organizations', 'Tone', 'Dates', 'GCAM', 'SharingImage',
       'RelatedImages', 'SocialImageEmbeds', 'SocialVideoEmbeds', 'Quotations',
       'AllNames', 'Amounts', 'TranslationInfo', 'Extras', 'GDELT_TIMESTAMP',
       'GDELT_URL', 'DATETIME'],
      dtype='object')

In [29]:
def parse_organizations(value: str):
    if pd.isna(value):
        return []

    return {
        org.rsplit(",", 1)[0].strip()
        for org in value.split(";")
        if org.strip()
    }

In [30]:
orgs = (
    df["V2Organizations"]
    .dropna()
    .apply(parse_organizations)
    .explode()
)

In [31]:
orgs.value_counts().head(20)

V2Organizations
United States         12838
White House            7330
Instagram              5608
Associated Press       5471
Facebook               3596
Google                 2276
Youtube                2195
Supreme Court          2062
Cnn                    1935
Netflix                1726
New York Times         1565
Oval Office            1540
Reuters                1512
Nasdaq                 1392
Justice Department     1360
Twitter                1262
European Union         1200
Human Services         1182
United Nations         1153
Espn                   1132
Name: count, dtype: int64

In [32]:
Names = (
    df['Persons'].str.strip()
    .str.lower()
    .str.split(';')
    .explode()
)

In [33]:
Names.value_counts().head(20)

Persons
donald trump           14882
los angeles             4531
joe biden               2974
elon musk               1732
vladimir putin          1492
pete hegseth            1296
las vegas               1226
keir starmer            1214
marco rubio              939
jeffrey epstein          888
pam bondi                846
barack obama             841
robert f kennedy jr      834
karoline leavitt         811
volodymyr zelenskyy      801
kamala harris            754
taylor swift             746
benjamin netanyahu       732
gavin newsom             709
mark carney              658
Name: count, dtype: int64

### Finding Energy Specific Articles

#### Theme based Filtering

In [34]:
TARGET_THEMES = {
    "NATURAL_DISASTER",
    "FLOODING",
    "DROUGHT",
    "HURRICANE",
    "SEVERE_WEATHER",
    "WEATHER",
    "STORM",
    "WILDFIRE", 
    "ENERGY"
}


def has_target_theme(value):
    if pd.isna(value):
        return False

    themes = {
        t.strip().upper()
        for t in value.split(";")
    }

    return bool(themes & TARGET_THEMES)

In [35]:
energy_themes_df = df[
    df["Themes"].apply(has_target_theme)
]

In [36]:
print(f"Original: {len(df):,}")
print(f"Filtered: {len(energy_themes_df):,}")
print(f"Kept: {len(energy_themes_df)/len(df):.2%}")

Original: 171,090
Filtered: 14,716
Kept: 8.60%


In [37]:
sample = energy_themes_df.sample(10, random_state=42)

for i, row in sample.iterrows():
    print("=" * 100)
    print(row["DocumentIdentifier"])

    for theme in sorted(row["Themes"].split(";")):
        print(f"  {theme}")

    print()

https://www.hopestandard.com/local-news/sunshine-valley-wildfire-now-held-8225002
  
  CRISISLEX_CRISISLEXREC
  CRISISLEX_O01_WEATHER
  CRISISLEX_T01_CAUTION_ADVICE
  NATURAL_DISASTER
  NATURAL_DISASTER_WILDFIRE

https://www.falmouthpacket.co.uk/news/national/25429131.lammy-puts-pressure-israel-gaza-aid-access/
  
  AFFECT
  ARMEDCONFLICT
  CEASEFIRE
  CRISISLEX_C03_WELLBEING_HEALTH
  EPU_POLICY
  EPU_POLICY_GOVERNMENT
  EVACUATION
  FOOD_SECURITY
  GENERAL_GOVERNMENT
  GENERAL_HEALTH
  KIDNAP
  MEDICAL
  NATURAL_DISASTER
  NATURAL_DISASTER_FAMINE
  RELEASE_HOSTAGE
  TAX_AIDGROUPS
  TAX_AIDGROUPS_OXFAM
  TAX_ETHNICITY
  TAX_ETHNICITY_PALESTINIANS
  TAX_FNCACT
  TAX_FNCACT_BABIES
  TAX_FNCACT_MIDWIVES
  TAX_FNCACT_OFFICIAL
  TAX_FNCACT_SECRETARY
  TAX_FNCACT_WOMEN
  TAX_POLITICAL_PARTY
  TAX_POLITICAL_PARTY_HAMAS
  TAX_TERROR_GROUP
  TAX_TERROR_GROUP_HAMAS
  UNGP_CLEAN_WATER_SANITATION
  UNGP_FORESTS_RIVERS_OCEANS
  WATER_SECURITY
  WB_1331_HEALTH_TECHNOLOGIES
  WB_1350_PHARMACEUTICALS


In [38]:
sample = energy_themes_df.sample(10, random_state=42)

theme_df = (
    sample[["DocumentIdentifier", "Themes"]]
    .assign(
        Themes=lambda x: x["Themes"].str.split(";")
    )
    .explode("Themes")
)

theme_df

,DocumentIdentifier,Themes
113694,https://www.hopestandard.com/local-news/sunshi...,NATURAL_DISASTER
113694,https://www.hopestandard.com/local-news/sunshi...,NATURAL_DISASTER_WILDFIRE
113694,https://www.hopestandard.com/local-news/sunshi...,CRISISLEX_T01_CAUTION_ADVICE
113694,https://www.hopestandard.com/local-news/sunshi...,CRISISLEX_CRISISLEXREC
113694,https://www.hopestandard.com/local-news/sunshi...,CRISISLEX_O01_WEATHER
...,...,...
122128,https://newschannel20.com/news/entertainment/g...,WB_2445_NON_STATE_SECURITY_ACTORS
122128,https://newschannel20.com/news/entertainment/g...,NATURAL_DISASTER
122128,https://newschannel20.com/news/entertainment/g...,NATURAL_DISASTER_TWISTERS
122128,https://newschannel20.com/news/entertainment/g...,TAX_FNCACT_INSIDER


In [39]:
sample = energy_themes_df.sample(50, random_state=42)

themes = (
    sample["Themes"]
    .str.strip()
    .str.split(";")
    .explode()
    .value_counts()
    .head()
)

print(themes.to_string())

Themes
NATURAL_DISASTER          50
                          50
TAX_FNCACT                43
CRISISLEX_CRISISLEXREC    31
EPU_POLICY                25


In [40]:
theme_df

,DocumentIdentifier,Themes
113694,https://www.hopestandard.com/local-news/sunshi...,NATURAL_DISASTER
113694,https://www.hopestandard.com/local-news/sunshi...,NATURAL_DISASTER_WILDFIRE
113694,https://www.hopestandard.com/local-news/sunshi...,CRISISLEX_T01_CAUTION_ADVICE
113694,https://www.hopestandard.com/local-news/sunshi...,CRISISLEX_CRISISLEXREC
113694,https://www.hopestandard.com/local-news/sunshi...,CRISISLEX_O01_WEATHER
...,...,...
122128,https://newschannel20.com/news/entertainment/g...,WB_2445_NON_STATE_SECURITY_ACTORS
122128,https://newschannel20.com/news/entertainment/g...,NATURAL_DISASTER
122128,https://newschannel20.com/news/entertainment/g...,NATURAL_DISASTER_TWISTERS
122128,https://newschannel20.com/news/entertainment/g...,TAX_FNCACT_INSIDER


#### Organization Based Filtering

In [41]:
def parse_organizations(value: str) -> set[str]:
    if pd.isna(value):
        return set()

    return {
        org.rsplit(",", 1)[0].strip()
        for org in value.split(";")
        if org.strip()
    }

In [42]:
org_df = (
    df[["DocumentIdentifier", "V2Organizations"]]
    .dropna(subset=["V2Organizations"])
    .assign(
        organization=lambda x: x["V2Organizations"].apply(parse_organizations)
    )
    .explode("organization")
    .drop(columns=["V2Organizations"])
    .drop_duplicates()
)

In [43]:
def inspect_org(org: str, n: int = 10):
    article_ids = (
        org_df[org_df["organization"].str.upper() == org.upper()]
        ["DocumentIdentifier"]
        .drop_duplicates()
    )

    print(f"Organization: {org}")
    print(f"Articles: {len(article_ids):,}")

    sample_ids = article_ids.sample(
        min(n, len(article_ids)),
        random_state=42
    )

    sample = df[df["DocumentIdentifier"].isin(sample_ids)]

    for _, row in sample.iterrows():
        print("=" * 100)
        print(row["DocumentIdentifier"])
        print("\nThemes:")
        for theme in sorted(str(row["Themes"]).split(";")):
            print(f"  - {theme}")

        print("\nOrganizations:")
        for organization in sorted(parse_organizations(row["V2Organizations"])):
            print(f"  - {organization}")

In [44]:
inspect_org("NOAA", 10)

Organization: NOAA
Articles: 0


In [45]:
inspect_org("National Weather Service", 10)

Organization: National Weather Service
Articles: 1,017
https://www.wvik.org/2025-01-15/la-fires-are-still-raging-but-forecasters-expect-calmer-winds-in-the-coming-days

Themes:
  - 
  - AFFECT
  - CRISISLEX_C07_SAFETY
  - CRISISLEX_CRISISLEXREC
  - CRISISLEX_O01_WEATHER
  - CRISISLEX_O02_RESPONSEAGENCIESATCRISIS
  - CRISISLEX_T01_CAUTION_ADVICE
  - CRISISLEX_T02_INJURED
  - CRISISLEX_T03_DEAD
  - CRISISLEX_T11_UPDATESSYMPATHY
  - DISASTER_FIRE
  - ECON_HOUSING_PRICES
  - KILL
  - LEADER
  - MANMADE_DISASTER
  - MANMADE_DISASTER_WITHOUT_POWER
  - NATURAL_DISASTER
  - NATURAL_DISASTER_HIGH_WINDS
  - NATURAL_DISASTER_STRONG_WINDS
  - POWER_OUTAGE
  - TAX_ECON_PRICE
  - TAX_FNCACT
  - TAX_FNCACT_CHIEF
  - TAX_FNCACT_FIRE_CHIEF
  - TAX_FNCACT_MAYOR
  - TAX_FNCACT_METEOROLOGIST
  - TAX_FNCACT_OFFICIALS
  - TAX_WORLDFISH
  - TAX_WORLDFISH_BASS
  - UNGP_FORESTS_RIVERS_OCEANS

Organizations:
  - National Weather Service
https://article.wn.com/view/2025/01/15/Fire_tornadoes_pose_a_threat_in_Cali

In [46]:
inspect_org("FEMA", 10)

Organization: FEMA
Articles: 0


Grid Operators

In [47]:
inspect_org("Midcontinent Independent System Operator", 10)

Organization: Midcontinent Independent System Operator
Articles: 0


In [48]:
inspect_org("MISO", 10)

Organization: MISO
Articles: 0


In [49]:
GRID_ORG_KEYWORDS = [
    "ERCOT",
    "ELECTRIC RELIABILITY COUNCIL OF TEXAS",
    "PJM",
    "PJM INTERCONNECTION",
    "MIDCONTINENT INDEPENDENT SYSTEM OPERATOR",
    "MISO",
    "CALIFORNIA INDEPENDENT SYSTEM OPERATOR",
    "CAISO",
    "NEW YORK INDEPENDENT SYSTEM OPERATOR",
    "NYISO",
    "ISO NEW ENGLAND",
    "NERC",
    "FERC",
    "WECC",
]

UTILITY_KEYWORDS = [
    "ENERGY",
    "ELECTRIC",
    "POWER",
    "UTILITY",
    "GRID",
]

pattern = "|".join(re.escape(x) for x in GRID_ORG_KEYWORDS + UTILITY_KEYWORDS)

grid_related_orgs = org_df[
    org_df["organization"]
    .str.upper()
    .str.contains(pattern, regex=True, na=False)
]

In [50]:

grid_related_orgs

,DocumentIdentifier,organization
12,https://www.newsday.co.zw/local-news/article/2...,National Electricity Transmission Grid South A...
12,https://www.newsday.co.zw/local-news/article/2...,Africa Electricity Supply Commission Eskom
151,https://www.brecorder.com/news/40340531/nepra-...,Energy Task
151,https://www.brecorder.com/news/40340531/nepra-...,Power Division
151,https://www.brecorder.com/news/40340531/nepra-...,National Electric Power Regulatory Authority N...
...,...,...
170892,https://www.dailymail.co.uk/money/comment/arti...,Octopus Energy
170939,https://www.thestar.com.my/business/business-n...,Cypark Renewable Energy Sdn Bhd
170947,https://island.lk/a-long-lost-son-is-found/,National People Power
171026,https://markets.financialcontent.com/stocks/ar...,Energy Era


In [51]:
grid_org_counts = (
    grid_related_orgs["organization"]
    .value_counts()
    .reset_index()
)

grid_org_counts.columns = ["organization", "article_count"]

grid_org_counts.head(50)

,organization,article_count
0,Department Of Energy,245
1,Energy Department,125
2,Energy Information Administration,105
3,International Atomic Energy Agency,78
4,International Emergency Economic Powers,52
5,Energy Security,51
6,Energy United Kingdom,51
7,International Energy Agency,44
8,Energy Consumers Miatta Fahnbulleh,44
9,National Grid,43


In [52]:
grid_article_ids = grid_related_orgs["DocumentIdentifier"].unique()

grid_articles_df = df[
    df["DocumentIdentifier"].isin(grid_article_ids)
]

In [53]:
for _, row in grid_articles_df.sample(10, random_state=42).iterrows():
    print("=" * 100)
    print(row["DocumentIdentifier"])

    print("\nThemes:")
    for theme in sorted(str(row["Themes"]).split(";")):
        print(f"  - {theme}")

    print("\nOrganizations:")
    for org in sorted(parse_organizations(row["V2Organizations"])):
        print(f"  - {org}")

https://www.wellingtontimes.com.au/story/9102196/no-silver-bullet-but-clean-energy-co-ownership-has-legs/

Themes:
  - 
  - AFFECT
  - AGRICULTURE
  - ENV_WINDPOWER
  - EPU_ECONOMY_HISTORIC
  - EPU_POLICY
  - EPU_POLICY_POLICY
  - MANMADE_DISASTER_IMPLIED
  - MARITIME
  - MARITIME_INCIDENT
  - SCIENCE
  - SOC_INNOVATION
  - TAX_ETHNICITY
  - TAX_ETHNICITY_AUSTRALIAN
  - TAX_FNCACT
  - TAX_FNCACT_CHIEF
  - TAX_FNCACT_CHIEF_EXECUTIVE
  - TAX_FNCACT_CHIEF_EXECUTIVE_OFFICER
  - TAX_FNCACT_COORDINATOR
  - TAX_FNCACT_DIRECTORS
  - TAX_FNCACT_EXECUTIVE
  - TAX_FNCACT_EXECUTIVE_OFFICER
  - TAX_FNCACT_MAN
  - TAX_FNCACT_OFFICER
  - TAX_MILITARY_TITLE
  - TAX_MILITARY_TITLE_OFFICER
  - TAX_WORLDLANGUAGES
  - TAX_WORLDLANGUAGES_WIRADJURI
  - TAX_WORLDMAMMALS
  - TAX_WORLDMAMMALS_CATS
  - UNGP_FORESTS_RIVERS_OCEANS
  - USPEC_POLICY1
  - WB_1377_ECOSYSTEM_ACCESS_AND_BENEFIT_SHARING
  - WB_1458_HEALTH_PROMOTION_AND_DISEASE_PREVENTION
  - WB_1462_WATER_SANITATION_AND_HYGIENE
  - WB_1699_METAL_ORE_MIN

### URL Based Filtering

In [54]:
URL_KEYWORDS = {
    "storm",
    "flood",
    "flooding",
    "hurricane",
    "tornado",
    "wildfire",
    "blizzard",
    "outage",
    "blackout",
    "power-outage",
    "power_outage",
    "without-power",
    "power-grid",
    "grid-congestion",  # typo fixed
}

def is_weather_url(url: str) -> bool:
    if pd.isna(url):
        return False

    url = url.lower()

    return any(
        keyword in url
        for keyword in URL_KEYWORDS
    )

In [55]:
url_articles_df = df[
    df["DocumentIdentifier"].apply(is_weather_url)
]

In [56]:
url_articles_df

,GKGRECORDID,DATE,SourceCollectionIdentifier,SourceCommonName,DocumentIdentifier,Counts,V2Counts,Themes,V2Themes,Locations,...,SocialImageEmbeds,SocialVideoEmbeds,Quotations,AllNames,Amounts,TranslationInfo,Extras,GDELT_TIMESTAMP,GDELT_URL,DATETIME
9,20250101000000-9,20250101000000,1,utv44.com,https://utv44.com/news/local/limited-tornado-d...,None,None,NATURAL_DISASTER;NATURAL_DISASTER_TORNADO;CRIS...,"EPU_ECONOMY_HISTORIC,1635;NATURAL_DISASTER_TOR...",None,...,None,https://youtube.com/channel/UCJc7is6F67EepYaJj...,None,"Mobile County Public Works,27;Mobile County,13...","2,tornados touched down,164;",None,<PAGE_AUTHORS>Keith Lane</PAGE_AUTHORS><PAGE_T...,20250101000000,http://data.gdeltproject.org/gdeltv2/202501010...,2025-01-01 00:00:00
282,20250101000000-282,20250101000000,1,fox10phoenix.com,https://www.fox10phoenix.com/news/puerto-rico-...,None,None,SECURITY_SERVICES;TAX_FNCACT;TAX_FNCACT_POLICE...,"TAX_FNCACT_OPERATOR,1338;TAX_AIDGROUPS_FEDERAL...","3#Palo Seco, Puerto Rico, United States#US#USP...",...,None,https://youtube.com/fox10phoenix;,None,"San Juan,106;Getty Images,248;New Year,412;Ass...","3000000,clients,247;47000000,clients,684;4,sto...",None,<PAGE_LINKS>https://www.foxnews.com/category/a...,20250101000000,http://data.gdeltproject.org/gdeltv2/202501010...,2025-01-01 00:00:00
388,20250101000000-388,20250101000000,1,wwmt.com,https://wwmt.com/news/local/top-stories-2024-m...,None,None,CRISISLEX_CRISISLEXREC;NATURAL_DISASTER;NATURA...,"TAX_FNCACT_SECRETARY_OF_STATE,1039;ENV_SOLAR,1...",1#United States#US#US#39.828175#-98.5795#US;2#...,...,None,https://youtube.com/user/wwmtnews;,None,"News Channel,289;West Michigan,450;West Michig...","3,top stories of 2024,228;100,of minimum wage ...",None,<PAGE_AUTHORS>Katie Sergent | News Channel 3</...,20250101000000,http://data.gdeltproject.org/gdeltv2/202501010...,2025-01-01 00:00:00
403,20250101000000-403,20250101000000,1,arlnow.com,https://www.arlnow.com/2024/12/31/just-in-seve...,None,None,CRISISLEX_O01_WEATHER;CRISISLEX_T01_CAUTION_AD...,"CRISISLEX_T01_CAUTION_ADVICE,102;CRISISLEX_T01...","3#King George County, Virginia, United States#...",...,https://pic.twitter.com/2NRwOW0f6h;,https://youtube.com/user/arlingtonnews/videos;,None,"Severe Thunderstorm,110;National Weather,269;N...","8,miles east of Quantico,1000;",None,<PAGE_LINKS>https://t.co/2NRwOW0f6h;https://tw...,20250101000000,http://data.gdeltproject.org/gdeltv2/202501010...,2025-01-01 00:00:00
429,20250101000000-429,20250101000000,1,castanetkamloops.net,https://www.castanetkamloops.net/news/BC/52530...,None,None,WB_569_HYDROMET_SERVICES;WB_568_CLIMATE_SERVIC...,"TAX_WORLDLANGUAGES_CHEYENNE,1062;UNGP_FORESTS_...","4#Vancouver, British Columbia, Canada#CA#CA02#...",...,None,None,"1242|43||Carson City , Oklahoma City , and Ral...","National Oceanic,32;Atmospheric Administration...",None,None,<PAGE_LINKS>https://weather.gc.ca/astro/clds_v...,20250101000000,http://data.gdeltproject.org/gdeltv2/202501010...,2025-01-01 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170791,20251231004500-737,20251231004500,1,cnbc.com,https://www.cnbc.com/2025/12/30/ro-khanna-call...,None,None,WB_2024_ANTI_CORRUPTION_AUTHORITIES;WB_696_PUB...,"CORRUPTION,100;WB_2019_ANTI_CORRUPTION_LEGISLA...","2#California, United States#US#USCA#36.17#-119...",...,None,https://youtube.com/user/cnbc/;https://youtube...,None,"Silicon Valley,244",None,None,<PAGE_AUTHORS>Garrett Downs</PAGE_AUTHORS><PAG...,20251231004500,http://data.gdeltproject.org/gdeltv2/202512310...,2025-12-31 00:45:00
170980,20251231004500-926,20251231004500,1,thewestmorlandgazette.co.uk,https://www.thewestmorlandgazette.co.uk/news/n...,"AFFECT#24##4#Dagenham, Barking And Dagenham, U...","AFFECT#24##4#Dagenham, Barking And Dagenham, U...",WATER_SECURITY;NATURAL_DISASTER;NATURAL_DISAST...,"NATURAL_DISASTER_WILDFIRES,268;NATURAL_DISASTE...","4#Ilford, Redbridge, United Kingdom#UK#UKK8#51...",...,None,None,415|65||exceptional bravery and profe

In [57]:
from collections import Counter

keyword_counts = Counter()

for url in df["DocumentIdentifier"].dropna():
    url = url.lower()

    for keyword in URL_KEYWORDS:
        if keyword in url:
            keyword_counts[keyword] += 1

pd.Series(keyword_counts).sort_values(ascending=False)

storm            925
flood            573
wildfire         539
hurricane        240
flooding         200
tornado          179
outage           152
blackout          76
power-outage      51
without-power     36
blizzard          13
power-grid         5
dtype: int64

In [58]:
def inspect_keyword(keyword, n=10):
    matches = df[
        df["DocumentIdentifier"]
        .str.lower()
        .str.contains(keyword, na=False)
    ]

    print(f"{keyword}: {len(matches):,} matches")

    return matches[
        ["DocumentIdentifier", "Themes"]
    ].sample(
        min(n, len(matches)),
        random_state=42
    )

In [59]:
inspect_keyword("outage")

outage: 152 matches


,DocumentIdentifier,Themes
78881,https://www.wsiu.org/2025-05-15/denver-air-tra...,TAX_FNCACT;TAX_FNCACT_PILOTS;SOC_POINTSOFINTER...
165066,https://www.nhpr.org/2025-12-15/waterbury-wate...,WB_137_WATER;CRISISLEX_C06_WATER_SANITATION;MA...
109763,https://www.clarksvilleonline.com/2025/08/15/c...,WB_137_WATER;AFFECT;ENV_NATURALGAS;WB_139_SANI...
90621,https://929zzu.com/2025/07/15/gusty-winds-lead...,SHORTAGE;MANMADE_DISASTER;MANMADE_DISASTER_POW...
139409,http://www.t-g.com/premium/newsusa/stories/how...,NATURAL_DISASTER;NATURAL_DISASTER_SEVERE_WEATH...
151971,https://krcrtv.com/news/local/butte-county-she...,TAX_FNCACT;TAX_FNCACT_SHERIFF;CRISISLEX_C07_SA...
44371,https://www.wpxi.com/news/local/outage-causes-...,PUBLIC_TRANSPORT;WB_1428_INJURY;WB_1406_DISEAS...
11284,https://www.illawarramercury.com.au/story/8867...,TAX_FNCACT;TAX_FNCACT_HUNTER;MANMADE_DISASTER;...
22662,https://www.cowichanvalleycitizen.com/news/wea...,SHORTAGE;NATURAL_DISASTER;NATURAL_DISASTER_POW...
17588,https://www.westhawaiitoday.com/2025/01/30/haw...,WB_137_WATER;WATER_SECURITY;WB_1199_WATER_SUPP...


In [60]:
inspect_keyword("hurricane")

hurricane: 240 matches


,DocumentIdentifier,Themes
58820,https://www.sun-sentinel.com/2025/04/14/hurric...,NATURAL_DISASTER;NATURAL_DISASTER_HURRICANES;C...
26457,https://b1039.com/2025/02/14/beach-closures-af...,None
120692,https://wcti12.com/news/local/hurricane-floren...,NATURAL_DISASTER;NATURAL_DISASTER_HURRICANE;CR...
125990,https://eagle1005.com/news/030030-1-man-killed...,TAX_FNCACT;TAX_FNCACT_AUTHORITIES;CRISISLEX_CR...
124104,https://www.wapt.com/article/government-shutdo...,UNREST_BELLIGERENT;GENERAL_GOVERNMENT;EPU_POLI...
142334,https://www.tenterfieldstar.com.au/story/91014...,NATURAL_DISASTER;NATURAL_DISASTER_HURRICANE;CR...
163853,https://www.orlandosentinel.com/2025/12/15/dav...,None
111656,https://www.timesleader.com/news/1713301/heavy...,NATURAL_DISASTER;NATURAL_DISASTER_HURRICANE;CR...
33303,https://www.yourobserver.com/news/2025/feb/27/...,TAX_FNCACT;TAX_FNCACT_BOARD_MEMBERS;TAX_FNCACT...
139992,http://www.jamaicantimes.com/news/278665419/hu...,NATURAL_DISASTER;NATURAL_DISASTER_HURRICANE;CR...


In [61]:
inspect_keyword("power-grid")

power-grid: 5 matches


,DocumentIdentifier,Themes
89606,https://www.renewableenergyworld.com/power-gri...,TAX_FNCACT;TAX_FNCACT_MODERATOR;WB_2745_JOB_QU...
134311,https://www.northcentralpa.com/features/spotli...,TAX_ECON_PRICE;ECON_ELECTRICALPRICE;ECON_ELECT...
101938,https://www.kgou.org/2025-07-31/how-utilities-...,ECON_ELECTRICALGRID;SHORTAGE;
85880,https://jamaica-gleaner.com/article/news/20250...,LEADER;TAX_FNCACT;TAX_FNCACT_GOVERNOR;USPEC_PO...
112029,https://www.theverge.com/news/760253/tech-ai-p...,ARMEDCONFLICT;EPU_POLICY;EPU_POLICY_POLITICAL;


### Relevence Scoring

In [62]:
TARGET_THEMES = {
    "WEATHER",
    "NATURAL_DISASTER",
    "FLOODING",
    "DROUGHT",
    "HURRICANE",
    "STORM",
    "WILDFIRE",
    "EXTREME_WEATHER",
}

WEATHER_ORG_KEYWORDS = [
    "NOAA",
    "NATIONAL WEATHER SERVICE",
    "NATIONAL HURRICANE CENTER",
    "FEMA",
    "EMERGENCY MANAGEMENT",
    "METEOROLOG",
    "DISASTER",
]

GRID_ORG_KEYWORDS = [
    "ERCOT",
    "PJM",
    "MISO",
    "CAISO",
    "NYISO",
    "ISO NEW ENGLAND",
    "NERC",
    "FERC",
    "WECC",
    "ELECTRIC",
    "POWER",
    "UTILITY",
    "GRID",
]

URL_KEYWORDS = {
    "storm", "flood", "flooding", "hurricane", "tornado",
    "wildfire", "blizzard", "outage", "blackout",
    "power-outage", "power_outage", "without-power",
    "power-grid", "grid-congestion",
}


def parse_themes(value: str) -> set[str]:
    if pd.isna(value):
        return set()

    return {
        t.strip().upper()
        for t in value.split(";")
        if t.strip()
    }


def parse_organizations(value: str) -> set[str]:
    if pd.isna(value):
        return set()

    return {
        org.rsplit(",", 1)[0].strip().upper()
        for org in value.split(";")
        if org.strip()
    }


def contains_any_keyword(text: str, keywords) -> bool:
    if pd.isna(text):
        return False

    text = str(text).upper()

    return any(keyword.upper() in text for keyword in keywords)


def has_target_theme(value: str) -> bool:
    themes = parse_themes(value)
    return bool(themes & TARGET_THEMES)


def has_weather_org(value: str) -> bool:
    orgs = parse_organizations(value)
    return any(
        contains_any_keyword(org, WEATHER_ORG_KEYWORDS)
        for org in orgs
    )


def has_grid_org(value: str) -> bool:
    orgs = parse_organizations(value)
    return any(
        contains_any_keyword(org, GRID_ORG_KEYWORDS)
        for org in orgs
    )


def has_url_keyword(url: str) -> bool:
    if pd.isna(url):
        return False

    url = str(url).lower()

    return any(keyword in url for keyword in URL_KEYWORDS)


def has_us_location(value: str) -> bool:
    if pd.isna(value):
        return False

    for loc in value.split(";"):
        parts = loc.split("#")

        if len(parts) >= 3:
            country_code = parts[2]

            if country_code == "US":
                return True

    return False


def add_relevance_scores(df: pd.DataFrame) -> pd.DataFrame:
    scored = df.copy()

    scored["theme_match"] = scored["Themes"].apply(has_target_theme)
    scored["weather_org_match"] = scored["V2Organizations"].apply(has_weather_org)
    scored["grid_org_match"] = scored["V2Organizations"].apply(has_grid_org)
    scored["url_keyword_match"] = scored["DocumentIdentifier"].apply(has_url_keyword)
    scored["us_location_match"] = scored["Locations"].apply(has_us_location)


    scored = scored.loc[scored['us_location_match']]
    
    scored["relevance_score"] = (
        5 * scored["theme_match"].astype(int)
        + 4 * scored["url_keyword_match"].astype(int)
        + 3 * scored["weather_org_match"].astype(int)
        + 10 * scored["grid_org_match"].astype(int)
    )

    return scored

In [63]:
scored_df = add_relevance_scores(df)

candidates = scored_df[
    scored_df["relevance_score"] >= 5
].sort_values("relevance_score", ascending=False)

candidates[
    [
        "DocumentIdentifier",
        "relevance_score",
        "theme_match",
        "url_keyword_match",
        "weather_org_match",
        "grid_org_match",
        "us_location_match",
        "Themes",
    ]
].head(5)

,DocumentIdentifier,relevance_score,theme_match,url_keyword_match,weather_org_match,grid_org_match,us_location_match,Themes
282,https://www.fox10phoenix.com/news/puerto-rico-...,22,True,True,True,True,True,SECURITY_SERVICES;TAX_FNCACT;TAX_FNCACT_POLICE...
9707,https://www.hometownregister.com/news/national...,22,True,True,True,True,True,NATURAL_DISASTER;NATURAL_DISASTER_WILDFIRES;CR...
157458,https://kprcradio.iheart.com/content/2025-11-2...,22,True,True,True,True,True,NATURAL_DISASTER;NATURAL_DISASTER_SEVERE_WEATH...
170616,https://www.yourdailyglobe.com/story/2025/12/3...,22,True,True,True,True,True,ECON_ELECTRICALGENERATION;SHORTAGE;NATURAL_DIS...
10515,https://www.elpasoinc.com/news/national/how-to...,22,True,True,True,True,True,NATURAL_DISASTER;NATURAL_DISASTER_WILDFIRES;CR...


In [64]:
sample = (
    candidates
    .sample(25, random_state=42)
    .sort_values("relevance_score", ascending=False)
)

for i, (_, row) in enumerate(sample.iterrows(), start=1):
    print("=" * 120)

    print(f"[{i}] Score: {row['relevance_score']}")

    print("\nURL:")
    print(row["DocumentIdentifier"])

    print("\nSignals:")
    print(f"  Theme Match      : {row['theme_match']}")
    print(f"  URL Match        : {row['url_keyword_match']}")
    print(f"  Weather Org      : {row['weather_org_match']}")
    print(f"  Grid Org         : {row['grid_org_match']}")
    print(f"  US Location      : {row['us_location_match']}")

    print("\nThemes:")

    for theme in sorted(
        str(row["Themes"]).split(";")
    ):
        print(f"  - {theme}")

    print()

[1] Score: 15

URL:
https://www.wmra.org/2025-08-15/after-a-freeze-trump-administration-reluctantly-agrees-to-fund-ev-chargers

Signals:
  Theme Match      : True
  URL Match        : False
  Weather Org      : False
  Grid Org         : True
  US Location      : True

Themes:
  - 
  - ALLIANCE
  - ARMEDCONFLICT
  - CRISISLEX_O01_WEATHER
  - CRISISLEX_T01_CAUTION_ADVICE
  - CRISISLEX_T09_DISPLACEDRELOCATEDEVACUATED
  - DELAY
  - ECON_WORLDCURRENCIES
  - ECON_WORLDCURRENCIES_DOLLAR
  - ENV_CLIMATECHANGE
  - ENV_GREEN
  - EPU_CATS_MIGRATION_FEAR_FEAR
  - EPU_CATS_REGULATION
  - EPU_POLICY
  - EPU_POLICY_CONGRESS
  - EPU_POLICY_GOVERNMENT
  - EPU_POLICY_LAW
  - EPU_POLICY_POLICY
  - EPU_POLICY_SPENDING
  - GENERAL_GOVERNMENT
  - LEADER
  - LEGISLATION
  - MANMADE_DISASTER_IMPLIED
  - NATURAL_DISASTER
  - NATURAL_DISASTER_EXTREME_WEATHER
  - RURAL
  - TAX_FNCACT
  - TAX_FNCACT_ATTORNEY
  - TAX_FNCACT_CEO
  - TAX_FNCACT_DIRECTOR
  - TAX_FNCACT_DRIVERS
  - TAX_FNCACT_EXECUTIVE
  - TAX_FNCACT

In [65]:

from datetime import datetime, timedelta

def timestamps_for_day(date: str, files_per_day: int = 96) -> list[str]:
    """
    GDELT GKG v2 files are available every 15 minutes.
    """
    start = datetime.strptime(date, "%Y-%m-%d")

    return [
        (start + timedelta(minutes=15 * i)).strftime("%Y%m%d%H%M%S")
        for i in range(files_per_day)
    ]

In [66]:
date= '2025-01-01'


timestamps = timestamps_for_day(date)

In [67]:
for i in timestamps:
    print(i)

20250101000000
20250101001500
20250101003000
20250101004500
20250101010000
20250101011500
20250101013000
20250101014500
20250101020000
20250101021500
20250101023000
20250101024500
20250101030000
20250101031500
20250101033000
20250101034500
20250101040000
20250101041500
20250101043000
20250101044500
20250101050000
20250101051500
20250101053000
20250101054500
20250101060000
20250101061500
20250101063000
20250101064500
20250101070000
20250101071500
20250101073000
20250101074500
20250101080000
20250101081500
20250101083000
20250101084500
20250101090000
20250101091500
20250101093000
20250101094500
20250101100000
20250101101500
20250101103000
20250101104500
20250101110000
20250101111500
20250101113000
20250101114500
20250101120000
20250101121500
20250101123000
20250101124500
20250101130000
20250101131500
20250101133000
20250101134500
20250101140000
20250101141500
20250101143000
20250101144500
20250101150000
20250101151500
20250101153000
20250101154500
20250101160000
20250101161500
2025010116

In [68]:
start = '2025-01-01'
end = '2025-01-04'


delta = pd.date_range(pd.to_datetime(start), pd.to_datetime(end))

In [69]:
delta

DatetimeIndex(['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04'], dtype='datetime64[ns]', freq='D')

In [70]:
records = df.to_dict(orient='records')

In [71]:
print(len(records))

171090


In [73]:
print(records[0])

{'GKGRECORDID': '20250101000000-0', 'DATE': '20250101000000', 'SourceCollectionIdentifier': '1', 'SourceCommonName': 'crooksandliars.com', 'DocumentIdentifier': 'https://crooksandliars.com/2024/12/retired-general-warns-musk-national', 'Counts': None, 'V2Counts': None, 'Themes': 'CRISISLEX_C07_SAFETY;WB_2470_PEACE_OPERATIONS_AND_CONFLICT_MANAGEMENT;WB_2432_FRAGILITY_CONFLICT_AND_VIOLENCE;WB_2490_NATIONAL_PROTECTION_AND_SECURITY;EPU_CATS_NATIONAL_SECURITY;MILITARY;TAX_MILITARY_TITLE;TAX_MILITARY_TITLE_LIEUTENANT;TAX_FNCACT;TAX_FNCACT_LIEUTENANT;TAX_MILITARY_TITLE_LIEUTENANT_GENERAL;TAX_FNCACT_LIEUTENANT_GENERAL;TAX_ETHNICITY;TAX_ETHNICITY_CHINESE;TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_CHINESE;TAX_FNCACT_COMMUNIST;USPEC_POLITICS_GENERAL1;TAX_POLITICAL_PARTY;TAX_POLITICAL_PARTY_COMMUNIST_PARTY;EPU_ECONOMY_HISTORIC;MANMADE_DISASTER_IMPLIED;TAX_FNCACT_LEADERS;GENERAL_GOVERNMENT;EPU_POLICY;EPU_POLICY_GOVERNMENT;LEGISLATION;TAX_FNCACT_CHIEF;', 'V2Themes': 'TAX_MILITARY_TITLE_LIEUTENANT,90;TAX_M